In [18]:
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from jaqalpaq import run
from jaqalpaq import emulator as jaqal_emulator_module
from jaqalpaq.run import (
    run_jaqal_file,
    run_jaqal_string,
    run_jaqal_batch,
    run_jaqal_circuit,
    frontend,
)

from jaqalpaq.emulator.unitary import UnitarySerializedEmulator
emulator_backend = UnitarySerializedEmulator()

from jaqalpaq.qsyntax import circuit
from jaqalpaq.generator import generate_jaqal_program
from jaqalpaq.core.algorithm import expand_subcircuits

from qscout.v1.std.noisy import SNLToy1

In [19]:
# Define your Jaqal circuit
jaqal_string = """from Calibration_PulseDefinitions.UserPulses usepulses *

let pi   3.1415926535897
let pi_2 1.57079
let num_gates 2
let ms_angle 1.57079

register q[2]
subcircuit{

loop num_gates{
    MS q[0] q[1] 0 ms_angle
}

}
"""

# Parity circuit: one calibrated MS gate prepares the Bell state, then an
# analysis pulse of variable phase is applied to both qubits.
parity_jaqal_string = """from Calibration_PulseDefinitions.UserPulses usepulses *

let num_gates 1
let ms_angle 1.5707963267948966
let analysis_phase 0.0
let analysis_angle 1.5707963267948966

register q[2]
subcircuit{
    loop num_gates{
        MS q[0] q[1] 0 ms_angle
    }
    R q[0] analysis_phase analysis_angle
    R q[1] analysis_phase analysis_angle
}
"""

In [20]:
NUM_TRIALS = 50
BASE_SEED = 20260808   # seeds the choice of TRIAL_SEEDS so a sweep is reproducible

# Physical bounds
AMP_IA_MIN,   AMP_IA_MAX   = 0.0,   100.0
ZETA_MIN,     ZETA_MAX     = 0.5,     1.5
SIDEBAND_OFFSET_MIN, SIDEBAND_OFFSET_MAX = -5000, 5000
CARRIER_OFFSET_MIN, CARRIER_OFFSET_MAX = -5000, 5000

# Gate parameters
NUM_GATES    = 2
MS_ANGLE     = math.pi / 2

# Run on the emulator (True) or on hardware (False).
#
# NOTE: the SNLToy1 error model is driven by amp_ia (rotation error) and zeta
# (depolarization) only. On the emulator, sideband_offset and carrier_offset
# have NO effect, so u[2] and u[3] are exactly flat directions and the
# recommended point will wander freely along them. That is expected in emulator
# mode, not a bug; on hardware all four parameters act through the pulse
# definitions.
emulator = True

# On the emulator, draw a real N-shot sample instead of returning the exact
# probability vector (see jaqal_objective). Keep True so that N actually means
# something in emulator mode; hardware runs sample by construction.
EMULATOR_SAMPLE_SHOTS = True

In [21]:
def jaqal_objective(u, N: int, sample=None):
    """
    u[0] -> amp_ia          in [0, 100]
    u[1] -> zeta            in [0.5, 1.5]
    u[2] -> sideband_offset in [-5000, 5000]
    u[3] -> carrier_offset  in [-5000, 5000]

    Returns population probabilities (p00, p01, p10, p11).
    Julia scores them as P(|11>) (see score_from_probs).

    `sample` (emulator only) overrides EMULATOR_SAMPLE_SHOTS: pass False to get
    the exact, noiseless population vector. That is what the deterministic
    reference trace uses -- the emulator analogue of q_det.
    """
    u = np.asarray(u, dtype=float)

    # Denormalize parameters (the surrogate model works on [-1, 1]^4)
    amp_ia   = AMP_IA_MIN   + (u[0] + 1.0) / 2.0 * (AMP_IA_MAX   - AMP_IA_MIN)
    zeta     = ZETA_MIN     + (u[1] + 1.0) / 2.0 * (ZETA_MAX     - ZETA_MIN)
    sideband_offset = SIDEBAND_OFFSET_MIN + (u[2] + 1.0) / 2.0 * (SIDEBAND_OFFSET_MAX - SIDEBAND_OFFSET_MIN)
    carrier_offset  = CARRIER_OFFSET_MIN  + (u[3] + 1.0) / 2.0 * (CARRIER_OFFSET_MAX  - CARRIER_OFFSET_MIN)

    # Simple error model based on Jaqal's SNLToy1. SNLToy1 multiplies the
    # rotation error by 10 for all two-qubit gates, likewise depolarization.
    delta_theta = (amp_ia * (np.pi / 80) - np.pi / 2) / 10
    depolarization = 2e-5 + 2e-2 * (zeta - 1.05) ** 2
    noisy_model = SNLToy1(2, rotation_error=delta_theta, depolarization=depolarization)

    if emulator:
        overrides = {
            "ms_angle":    MS_ANGLE,
            "num_gates":   NUM_GATES,
            "__repeats__": int(N),
        }
        # Emulator version swaps the pulse library for the standard gate set.
        jaqal_emulator = jaqal_string.replace(
            "from Calibration_PulseDefinitions.UserPulses usepulses *",
            "from qscout.v1.std usepulses *",
        )
        results = run_jaqal_string(jaqal_emulator, backend=noisy_model, overrides=overrides)
        subcircuit = results.subcircuits[0]
        # IMPORTANT: on the emulator `probability_by_int` is the EXACT probability
        # vector — it is bit-identical for __repeats__=10 and __repeats__=10000, so
        # using it makes the objective deterministic and N meaningless, while the
        # GP is still told the noise is sqrt(y(1-y)/N). `relative_frequency_by_int`
        # is the actual N-shot sample (integer counts / N) and is what hardware
        # returns, so it is the faithful stand-in. Set EMULATOR_SAMPLE_SHOTS=False
        # to recover the old (deterministic) behaviour.
        use_sample = EMULATOR_SAMPLE_SHOTS if sample is None else bool(sample)
        if use_sample:
            probs = subcircuit.relative_frequency_by_int
        else:
            probs = subcircuit.probability_by_int
    else:
        overrides = {
            "pd.amp_ia": amp_ia,
            "pd.zeta": zeta,
            "pd.sideband_offset": sideband_offset,
            "pd.carrier_offset": carrier_offset,
            "ms_angle": MS_ANGLE,
            "num_gates": NUM_GATES,
            "__repeats__": int(N),
        }
        results = run.run_jaqal_string(jaqal_string, overrides=overrides)
        probs = results.subcircuits[0].probability_by_int

    # Return the full population vector; Julia scores it as P(|11>).
    return np.asarray(probs, dtype=float)

In [22]:
# Load the Julia surrogate model and optimizer.
from juliacall import Main as jl

_repo_root = Path(os.getcwd()).resolve()
jl.seval(f'import Pkg; Pkg.activate("{_repo_root}"; io=devnull)')
jl.seval('Pkg.instantiate(; io=devnull)')

# bayes_hetero_opt.jl is self-contained: it carries the shared GP + BO core
# verbatim from the main branch plus the hardware-facing driver.
include_path = _repo_root / "src" / "bayes_hetero_opt.jl"
jl.seval(f'include("{include_path}")')

print("Julia BO loaded.")

┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API ~/.julia/juliaup/julia-1.12.6+0.aarch64.apple.darwin14/share/julia/stdlib/v1.12/Pkg/src/API.jl:1322


Julia BO loaded.


In [23]:
# --- Experiment sweep ---
N_LIST = [50, 100]                   # shots per evaluation (matches __repeats__ in Jaqal)
random.seed(BASE_SEED)               # so re-running the sweep reproduces it exactly
TRIAL_SEEDS = random.sample(range(10000), NUM_TRIALS)  # the same seeds are reused for every N
EXPERIMENTS_PER_N = NUM_TRIALS
TARGET_POPULATION_INDEX = 3          # p11 in population order p00, p01, p10, p11

# --- BO tunables (defaults match bayesopt_ucb on the main branch) ---
N_INIT = 12                          # Latin-hypercube evaluations before the GP is fitted
N_ITER = 75                          # BO iterations after initialisation
KAPPA = 1.96                         # UCB exploration weight
HYPER_EVERY = 10                     # re-optimise GP hyperparameters every N iterations
N_RESTARTS = 6                       # multi-start restarts for hyperparameter optimisation
M_ACQ = 100000                       # UCB acquisition candidates. A uniform scan degrades
                                     # sharply with dimension; at d=4 raising this from 20k to
                                     # 100k improved final 1-Q from 0.0093 to 0.0060 (16 seeds).
M_REC = 20000                        # recommendation scan candidates, refined by bounded L-BFGS
FIDELITY_THRESH = None               # early-stop when measured score >= threshold, e.g. 0.98
LEARN_NOISE_SCALE = True
JITTER = 1e-8                        # GP Cholesky jitter

# --- Saved GP slices and validation scan ---
GP_SLICE_ITERS = [10, 30, 50]
GP_SLICE_AXES = {"amp_ia": 0, "sideband_offset": 2}
GP_SLICE_GRID_SIZE = 201
PARITY_NUM_GATES = 1                 # one calibrated MS gate prepares the Bell state
PARITY_PHASES = np.linspace(0.0, np.pi, 33)
PARITY_TRIAL_STRIDE = 5
PARITY_CHECKPOINT_TRIALS = tuple(range(1, NUM_TRIALS + 1, PARITY_TRIAL_STRIDE))

# Normalised bounds [-1, 1]^4 passed to Julia
bounds = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)]
jl_bounds = jl.seval("Vector{Tuple{Float64,Float64}}")([tuple(b) for b in bounds])

In [24]:
# Julia BO wrapper + N-sweep experiment
jl.seval("""
function _run_bo(f, bounds, n_shots, n_init, n_iter, kap, seed_val,
                 hyper_every, n_restarts, m_acq, m_rec,
                 fid_thresh, learn_noise, jitter)
    bayesopt_ucb_threshold(f;
        bounds=bounds,
        n_shots=n_shots,
        n_init=n_init,
        n_iter=n_iter,
        M_acq=m_acq,
        M_rec=m_rec,
        κ=kap,
        seed=seed_val,
        maximize=true,
        fidelity_threshold=fid_thresh,
        hyper_every=hyper_every,
        learn_noise_scale=learn_noise,
        n_restarts=n_restarts,
        jitter=jitter,
    )
end
""")

def u_to_physical(u):
    u = np.asarray(u, dtype=float)
    amp_ia = AMP_IA_MIN + (u[0] + 1.0) / 2.0 * (AMP_IA_MAX - AMP_IA_MIN)
    zeta = ZETA_MIN + (u[1] + 1.0) / 2.0 * (ZETA_MAX - ZETA_MIN)
    sideband_offset = SIDEBAND_OFFSET_MIN + (u[2] + 1.0) / 2.0 * (SIDEBAND_OFFSET_MAX - SIDEBAND_OFFSET_MIN)
    carrier_offset = CARRIER_OFFSET_MIN + (u[3] + 1.0) / 2.0 * (CARRIER_OFFSET_MAX - CARRIER_OFFSET_MIN)
    return amp_ia, zeta, sideband_offset, carrier_offset

def result_array(value, dtype=float):
    return np.asarray(value, dtype=dtype).copy()

def score_from_populations(probs):
    """Calibration score = P(|11>), matching score_from_probs in Julia and
    Q_varMS (which returns w[4]) on the main branch."""
    probs = np.asarray(probs, dtype=float)
    return float(np.clip(probs[TARGET_POPULATION_INDEX], 0.0, 1.0))

def _plain(value):
    """Plain-Python view of a value, for DataFrame.attrs.

    pandas compares `attrs` dicts with `==` whenever it truncates a frame for
    display or concatenates one (DataFrame.__finalize__). A numpy array in there
    makes that comparison ambiguous and raises
    "The truth value of an array with more than one element is ambiguous",
    e.g. on a bare `df.head()`. Keep attrs free of arrays.
    """
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return [_plain(v) for v in value]
    if isinstance(value, dict):
        return {k: _plain(v) for k, v in value.items()}
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    return value

def experiment_metadata():
    return _plain({
        "N_LIST": list(N_LIST),
        "base_seed": BASE_SEED,
        "trial_seeds": list(TRIAL_SEEDS),
        "score": "p11",
        "n_init": N_INIT,
        "n_iter": N_ITER,
        "kappa": KAPPA,
        "hyper_every": HYPER_EVERY,
        "n_restarts": N_RESTARTS,
        "m_acq": M_ACQ,
        "m_rec": M_REC,
        "learn_noise_scale": LEARN_NOISE_SCALE,
        "jitter": JITTER,
        "emulator": bool(emulator),
        "emulator_sample_shots": bool(EMULATOR_SAMPLE_SHOTS),
        "gp_slice_iters": list(GP_SLICE_ITERS),
        "gp_slice_axes": dict(GP_SLICE_AXES),
        "parity_num_gates": PARITY_NUM_GATES,
        "parity_phases": PARITY_PHASES,
        "parity_trial_stride": PARITY_TRIAL_STRIDE,
        "parity_checkpoint_trials": list(PARITY_CHECKPOINT_TRIALS),
        "parity_selection_population": "p11",
        "parity_selection_rule": "final_recommended",   # x_rec; see collect_parity_checkpoint
        "parity_selection_point": "x_rec",
        "parameter_order": ["amp_ia", "zeta", "sideband_offset", "carrier_offset"],
        "population_order": ["p00", "p01", "p10", "p11"],
    })

def save_results(runs):
    saved_df = pd.DataFrame(runs)
    saved_df.attrs["metadata"] = experiment_metadata()
    saved_df.to_pickle(RESULTS_PATH)
    return saved_df

def run_parity_scan(u, N, phases, use_emulator=False):
    amp_ia, zeta, sideband_offset, carrier_offset = u_to_physical(u)
    delta_theta = (amp_ia * (np.pi / 80.0) - np.pi / 2.0) / 10.0
    depolarization = 2e-5 + 2e-2 * (zeta - 1.05) ** 2
    noisy_model = SNLToy1(2, rotation_error=delta_theta, depolarization=depolarization)
    circuit_string = parity_jaqal_string
    if use_emulator:
        circuit_string = circuit_string.replace(
            "from Calibration_PulseDefinitions.UserPulses usepulses *",
            "from qscout.v1.std usepulses *",
        )

    populations = []
    for phase in phases:
        overrides = {
            "ms_angle": MS_ANGLE,
            "num_gates": PARITY_NUM_GATES,
            "analysis_phase": float(phase),
            "analysis_angle": np.pi / 2.0,
            "__repeats__": int(N),
        }
        if use_emulator:
            result = run_jaqal_string(circuit_string, backend=noisy_model, overrides=overrides)
        else:
            overrides.update({
                "pd.amp_ia": amp_ia,
                "pd.zeta": zeta,
                "pd.sideband_offset": sideband_offset,
                "pd.carrier_offset": carrier_offset,
            })
            result = run.run_jaqal_string(circuit_string, overrides=overrides)
        sub = result.subcircuits[0]
        populations.append(np.asarray(
            sub.relative_frequency_by_int if (use_emulator and EMULATOR_SAMPLE_SHOTS)
            else sub.probability_by_int, dtype=float))

    populations = np.asarray(populations, dtype=float)
    parity = populations[:, 0] + populations[:, 3] - populations[:, 1] - populations[:, 2]
    return {
        "backend": "emulator" if use_emulator else "hardware",
        "N": int(N),
        "phases": np.asarray(phases, dtype=float),
        "populations": populations,
        "parity": parity,
        "recommended_u": np.asarray(u, dtype=float),
        "recommended_physical": np.asarray(u_to_physical(u), dtype=float),
    }

def collect_parity_checkpoint(run_record):
    # Scan the BO's final recommendation `x_rec`; the measured argmax overfits
    # shot noise at small N. Selecting argmax_n p11(x_n*) over a noisy trace is an
    # extreme-value statistic: it is upward-biased at the point it picks, and when
    # the measurement saturates (1 - p11 well below 1/N) most iterations tie at
    # p11 = 1.0 and np.nanargmax returns the FIRST of them -- an early, essentially
    # uncalibrated recommendation. x_rec is also the point the optimizer actually
    # returns, so the parity scan characterises the gate you would deploy.
    #
    # Note x_rec != X_opt[:, -1]: x_rec comes from the extra GP refit after the
    # loop (n_restarts + 2 restarts), matching what bayesopt_ucb returns.
    parity_u = np.asarray(run_record["x_rec"], dtype=float)
    run_record["is_parity_checkpoint"] = True
    # x_rec is produced after the final iteration; it is not one of the X_opt columns.
    run_record["parity_selection_iteration"] = int(run_record["n_iter"])
    # measured P(|11>) at x_rec == result.y_last_opt == the `score` column
    run_record["parity_selection_population"] = float(run_record["score"])
    run_record["parity_recommended_u"] = parity_u
    run_record["parity_recommended_physical"] = np.asarray(u_to_physical(parity_u), dtype=float)
    scan = run_parity_scan(parity_u, run_record["N"], PARITY_PHASES, use_emulator=emulator)
    run_record["parity_emulator" if emulator else "parity_hardware"] = scan
    return run_record

def record_run(result, n_shots, trial_index, seed, bo_elapsed):
    X = result_array(result.X)
    y = result_array(result.y)
    sigma_y = result_array(result.σy)
    i_acq = result_array(result.i_acq, dtype=int)
    i_opt = result_array(result.i_opt, dtype=int)
    X_opt = result_array(result.X_opt)
    X_acq = X[:, i_acq - 1]  # Julia indices are 1-based
    physical_acq_history = np.asarray([
        u_to_physical(X_acq[:, k]) for k in range(X_acq.shape[1])
    ], dtype=float)

    # Re-query every per-iteration recommendation so the convergence trace is
    # the optimum as measured by the backend, not the GP's belief about it.
    population_t0 = time.time()
    recommended_populations = np.vstack([
        np.asarray(jaqal_objective(X_opt[:, k], n_shots), dtype=float)
        for k in range(X_opt.shape[1])
    ])
    # Exact (noiseless) populations at the same points: the emulator analogue of
    # q_det. Sampled traces bottom out at the 1/N resolution floor, so this is
    # what actually shows convergence.
    if emulator:
        recommended_populations_exact = np.vstack([
            np.asarray(jaqal_objective(X_opt[:, k], n_shots, sample=False), dtype=float)
            for k in range(X_opt.shape[1])
        ])
    else:
        recommended_populations_exact = None
    population_elapsed = time.time() - population_t0
    physical_history = np.asarray([
        u_to_physical(X_opt[:, k]) for k in range(X_opt.shape[1])
    ], dtype=float)

    # x_rec is the final recommendation after the extra GP refit (same step
    # bayesopt_ucb performs); X_opt[:, -1] is the last in-loop recommendation.
    x_rec = result_array(result.x_rec)
    amp_last, zeta_last, sideband_last, carrier_last = u_to_physical(x_rec)
    return {
        "N": int(n_shots),
        "trial": int(trial_index),
        "seed": int(seed),
        "score": float(result.y_last_opt),
        "mu_last": float(result.mu_opt[-1]),
        "m_rec": float(result.m_rec),
        "n_iter": int(result.n_iter_actual),
        "total_shots": int(result.total_shots),
        "population_history_shots": int(n_shots * X_opt.shape[1]),
        "elapsed_s": float(bo_elapsed),
        "population_elapsed_s": float(population_elapsed),
        "amp_ia": float(amp_last),
        "zeta": float(zeta_last),
        "sideband_offset": float(sideband_last),
        "carrier_offset": float(carrier_last),
        "X": X,
        "y": y,
        "sigma_y": sigma_y,
        "i_acq": i_acq,
        "i_opt": i_opt,
        "X_acq": X_acq,
        "physical_acq_history": physical_acq_history,
        "X_opt": X_opt,
        "x_rec": x_rec,
        "mu_opt": result_array(result.mu_opt),
        "theta_i": result_array(result.θ_i),
        "physical_history": physical_history,
        "recommended_populations": recommended_populations,
        "recommended_populations_exact": recommended_populations_exact,
        "gp_slices": {},
        "is_selected": False,
        "is_parity_checkpoint": False,
        "parity_selection_iteration": None,
        "parity_selection_population": None,
        "parity_recommended_u": None,
        "parity_recommended_physical": None,
        "parity_hardware": None,
        "parity_emulator": None,
        "emulator_validation": None,
    }

learn_noise_jl = bool(LEARN_NOISE_SCALE)
fid_julia = jl.nothing if FIDELITY_THRESH is None else float(FIDELITY_THRESH)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / ("bo_sweep_sandia_emulator.pkl" if emulator else "bo_sweep_sandia.pkl")

all_runs = []
for n_shots in N_LIST:
    for trial_index, seed in enumerate(TRIAL_SEEDS):
        np.random.seed(seed)
        t0 = time.time()
        result = jl._run_bo(
            jaqal_objective,
            jl_bounds,
            int(n_shots),
            int(N_INIT),
            int(N_ITER),
            float(KAPPA),
            int(seed),
            int(HYPER_EVERY),
            int(N_RESTARTS),
            int(M_ACQ),
            int(M_REC),
            fid_julia,
            learn_noise_jl,
            float(JITTER),
        )
        bo_elapsed = time.time() - t0
        rec = record_run(result, n_shots, trial_index, seed, bo_elapsed)
        trial_number = trial_index + 1
        if trial_number in PARITY_CHECKPOINT_TRIALS:
            collect_parity_checkpoint(rec)
            print(
                f"Collected parity checkpoint for N={n_shots}, trial={trial_number}, "
                f"p11={rec['parity_selection_population']:.6f}"
            )
        all_runs.append(rec)
        save_results(all_runs)  # checkpoint calibration and any interleaved parity data
        print(
            f"N={n_shots} trial={trial_number}/{EXPERIMENTS_PER_N} seed={seed} "
            f"score={rec['score']:.4f} BO_shots={rec['total_shots']} "
            f"iters={rec['n_iter']} time={bo_elapsed:.1f}s"
        )

Collected parity checkpoint for N=50, trial=1, p11=1.000000
N=50 trial=1/50 seed=8837 score=1.0000 BO_shots=4400 iters=75 time=27.7s
N=50 trial=2/50 seed=553 score=1.0000 BO_shots=4400 iters=75 time=40.5s
N=50 trial=3/50 seed=5638 score=0.9800 BO_shots=4400 iters=75 time=53.7s
N=50 trial=4/50 seed=2209 score=1.0000 BO_shots=4400 iters=75 time=49.5s
N=50 trial=5/50 seed=444 score=1.0000 BO_shots=4400 iters=75 time=51.3s
Collected parity checkpoint for N=50, trial=6, p11=0.960000
N=50 trial=6/50 seed=3979 score=0.9600 BO_shots=4400 iters=75 time=44.9s
N=50 trial=7/50 seed=8144 score=0.9400 BO_shots=4400 iters=75 time=42.9s
N=50 trial=8/50 seed=3370 score=1.0000 BO_shots=4400 iters=75 time=42.7s
N=50 trial=9/50 seed=546 score=1.0000 BO_shots=4400 iters=75 time=43.0s
N=50 trial=10/50 seed=7615 score=1.0000 BO_shots=4400 iters=75 time=44.8s
Collected parity checkpoint for N=50, trial=11, p11=1.000000
N=50 trial=11/50 seed=2212 score=1.0000 BO_shots=4400 iters=75 time=42.3s
N=50 trial=12/50 

In [25]:
# Reconstruct and save exact GP posterior slices from archived training data.
jl.seval("""
function _gp_slice_predict(X_train, y_train, sigma_train, theta, X_query, jitter)
    X_jl = Matrix{Float64}(X_train)
    y_jl = Vector{Float64}(y_train)
    sigma_jl = Vector{Float64}(sigma_train)
    theta_jl = Vector{Float64}(theta)
    query_jl = Matrix{Float64}(X_query)
    gp = fit_heterogp(X_jl, y_jl, sigma_jl;
                      θ_init=theta_jl,
                      learn_hypers=false,
                      learn_noise_scale=true,
                      jitter=Float64(jitter))
    mu = Vector{Float64}(undef, size(query_jl, 2))
    variance = similar(mu)
    for j in axes(query_jl, 2)
        mu[j], variance[j] = predict_latent(gp, Vector{Float64}(query_jl[:, j]))
    end
    return mu, variance
end
""")

def make_gp_slices(run_record):
    slices = {}
    x_reference = np.asarray(run_record["X_opt"][:, -1], dtype=float)
    i_acq_zero = np.asarray(run_record["i_acq"], dtype=int) - 1  # Julia indices are 1-based
    for axis_name, axis_index in GP_SLICE_AXES.items():
        slices[axis_name] = {}
        x_u = np.linspace(bounds[axis_index][0], bounds[axis_index][1], GP_SLICE_GRID_SIZE)
        X_query = np.repeat(x_reference[:, None], GP_SLICE_GRID_SIZE, axis=1)
        X_query[axis_index, :] = x_u
        x_physical = np.asarray([u_to_physical(X_query[:, j])[axis_index] for j in range(GP_SLICE_GRID_SIZE)])

        for snapshot_iter in GP_SLICE_ITERS:
            if snapshot_iter > run_record["n_iter"]:
                continue
            n_train = N_INIT + snapshot_iter - 1
            if len(i_acq_zero) < n_train:
                raise ValueError(f"Not enough acquisition points to reconstruct iteration {snapshot_iter}.")
            train_indices = i_acq_zero[:n_train]
            theta = np.asarray(run_record["theta_i"][:, snapshot_iter - 1], dtype=float)
            prediction = jl._gp_slice_predict(
                run_record["X"][:, train_indices],
                run_record["y"][train_indices],
                run_record["sigma_y"][train_indices],
                theta,
                X_query,
                float(JITTER),
            )
            mu = np.asarray(prediction[0], dtype=float)
            sigma = np.sqrt(np.maximum(np.asarray(prediction[1], dtype=float), 0.0))
            slices[axis_name][snapshot_iter] = {
                "N": run_record["N"],
                "trial": run_record["trial"],
                "seed": run_record["seed"],
                "iteration": snapshot_iter,
                "axis": axis_name,
                "axis_index": axis_index,
                "x_u": x_u.copy(),
                "x_physical": x_physical.copy(),
                "X_query": X_query.copy(),
                "fixed_u": x_reference.copy(),
                "fixed_physical": np.asarray(u_to_physical(x_reference), dtype=float),
                "training_indices": train_indices.copy(),
                "theta": theta.copy(),
                "mu": mu,
                "sigma": sigma,
            }
    return slices

# Keep one representative run per N for GP plotting only.
for run_record in all_runs:
    run_record["is_selected"] = False
for n_shots in N_LIST:
    candidates = [r for r in all_runs if r["N"] == n_shots]
    selected = max(
        candidates,
        key=lambda r: float(
            np.nanmax(np.asarray(r["recommended_populations"])[:, TARGET_POPULATION_INDEX])
        ),
    )
    selected["is_selected"] = True

for run_number, run_record in enumerate(all_runs, start=1):
    run_record["gp_slices"] = make_gp_slices(run_record)
    print(f"Saved GP slices for run {run_number}/{len(all_runs)} (N={run_record['N']}, seed={run_record['seed']}).")

df = save_results(all_runs)
print(f"Checkpointed GP slices at iterations {GP_SLICE_ITERS} to {RESULTS_PATH}.")

Saved GP slices for run 1/100 (N=50, seed=8837).
Saved GP slices for run 2/100 (N=50, seed=553).
Saved GP slices for run 3/100 (N=50, seed=5638).
Saved GP slices for run 4/100 (N=50, seed=2209).
Saved GP slices for run 5/100 (N=50, seed=444).
Saved GP slices for run 6/100 (N=50, seed=3979).
Saved GP slices for run 7/100 (N=50, seed=8144).
Saved GP slices for run 8/100 (N=50, seed=3370).
Saved GP slices for run 9/100 (N=50, seed=546).
Saved GP slices for run 10/100 (N=50, seed=7615).
Saved GP slices for run 11/100 (N=50, seed=2212).
Saved GP slices for run 12/100 (N=50, seed=9721).
Saved GP slices for run 13/100 (N=50, seed=1395).
Saved GP slices for run 14/100 (N=50, seed=8128).
Saved GP slices for run 15/100 (N=50, seed=5884).
Saved GP slices for run 16/100 (N=50, seed=921).
Saved GP slices for run 17/100 (N=50, seed=6768).
Saved GP slices for run 18/100 (N=50, seed=7484).
Saved GP slices for run 19/100 (N=50, seed=6209).
Saved GP slices for run 20/100 (N=50, seed=4795).
Saved GP slic

In [26]:
# Post-process emulator validation and GP reference curves without extra parity scans.
selected_runs = [r for r in all_runs if r["is_selected"]]
_emulator_before_validation = emulator
try:
    emulator = True
    for selected in selected_runs:
        u_reference = np.asarray(selected["X_opt"][:, -1], dtype=float)
        validation_populations = np.asarray(
            jaqal_objective(u_reference, selected["N"]), dtype=float
        )
        selected["emulator_validation"] = {
            "N": selected["N"],
            "populations": validation_populations,
            "score": score_from_populations(validation_populations),
            "recommended_u": u_reference,
            "recommended_physical": np.asarray(u_to_physical(u_reference), dtype=float),
        }

        for axis_name in GP_SLICE_AXES:
            first_snapshot = selected["gp_slices"][axis_name][GP_SLICE_ITERS[0]]
            X_query = first_snapshot["X_query"]
            reference_populations = np.vstack([
                np.asarray(jaqal_objective(X_query[:, j], selected["N"]), dtype=float)
                for j in range(X_query.shape[1])
            ])
            reference_scores = np.asarray([
                score_from_populations(probs) for probs in reference_populations
            ])
            for snapshot_iter in GP_SLICE_ITERS:
                slice_data = selected["gp_slices"][axis_name][snapshot_iter]
                slice_data["emulator_reference_populations"] = reference_populations.copy()
                slice_data["emulator_reference_score"] = reference_scores.copy()
        print(f"Completed emulator validation and GP references for N={selected['N']}.")
finally:
    emulator = _emulator_before_validation

df = save_results(all_runs)
print(f"Saved interleaved parity and GP data to {RESULTS_PATH}.")

Completed emulator validation and GP references for N=50.
Completed emulator validation and GP references for N=100.
Saved interleaved parity and GP data to results/bo_sweep_sandia_emulator.pkl.


In [27]:
# Aggregate final results and verify the matched-seed design.
df = save_results(all_runs)
summary = df.groupby("N").agg(
    score_mean=("score", "mean"),
    score_std=("score", "std"),
    shots_mean=("total_shots", "mean"),
    shots_std=("total_shots", "std"),
    iters_mean=("n_iter", "mean"),
    elapsed_mean=("elapsed_s", "mean"),
).reset_index()

# Verify the parity checkpoints really scanned x_rec.
checkpoints = [r for r in all_runs if r["is_parity_checkpoint"]]
for checkpoint in checkpoints:
    assert np.allclose(checkpoint["parity_recommended_u"], checkpoint["x_rec"])
    assert checkpoint["parity_selection_iteration"] == checkpoint["n_iter"]
    assert np.isclose(checkpoint["parity_selection_population"], checkpoint["score"])
print(f"{len(checkpoints)} parity checkpoints, all scanned at x_rec (final_recommended).")

seeds_per_N = df.groupby("N")["seed"].apply(lambda s: tuple(sorted(s)))
matched = len(set(seeds_per_N)) == 1
print(f"score=p11, {EXPERIMENTS_PER_N} trials per N, "
      f"matched seeds across N: {matched}\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nSaved: {RESULTS_PATH}")
df.head()

20 parity checkpoints, all scanned at x_rec (final_recommended).
score=p11, 50 trials per N, matched seeds across N: True

  N  score_mean  score_std  shots_mean  shots_std  iters_mean  elapsed_mean
 50      0.9932     0.0132   4400.0000     0.0000     75.0000       45.1363
100      0.9974     0.0056   8800.0000     0.0000     75.0000       43.3657

Saved: results/bo_sweep_sandia_emulator.pkl


,N,trial,seed,score,mu_last,m_rec,n_iter,total_shots,population_history_shots,elapsed_s,...,gp_slices,is_selected,is_parity_checkpoint,parity_selection_iteration,parity_selection_population,parity_recommended_u,parity_recommended_physical,parity_hardware,parity_emulator,emulator_validation
0,50,0,8837,1.00,1.017057,1.013480,75,4400,3750,27.732239,...,"{'amp_ia': {10: {'N': 50, 'trial': 0, 'seed': ...",True,True,75.0,1.0,"[-0.19493926321243515, 0.17367448782580616, 0....","[40.25303683937824, 1.086837243912903, 3925.91...",None,"{'backend': 'emulator', 'N': 50, 'phases': [0....","{'N': 50, 'populations': [0.0, 0.0, 0.0, 1.0],..."
1,50,1,553,1.00,1.012632,1.010026,75,4400,3750,40.469904,...,"{'amp_ia': {10: {'N': 50, 'trial': 1, 'seed': ...",False,False,NaN,NaN,None,None,None,None,None
2,50,2,5638,0.98,1.011200,1.020762,75,4400,3750,53.723986,...,"{'amp_ia': {10: {'N': 50, 'trial': 2, 'seed': ...",False,False,NaN,NaN,None,None,None,None,None
3,50,3,2209,1.00,1.015106,1.015168,75,4400,3750,49.515948,...,"{'amp_ia': {10: {'N': 50, 'trial': 3, 'seed': ...",False,False,NaN,NaN,None,None,None,None,None
4,50,4,444,1.00,1.016961,1.010521,75,4400,3750,51.282134,...,"{'amp_ia': {10: {'N': 50, 'trial': 4, 'seed': ...",False,False,NaN,NaN,None,None,None,None,None
